# Домашка: контрастивный лосс CLIP

Это практическое продолжение семинара 13 (ViT / CLIP). Вы реализуете
**контрастивный лосс CLIP**, своими руками обучите мини-CLIP поверх замороженных
крошечных предобученных энкодеров, а затем используете его для **zero-shot
классификации** картинок.

⚠️ В лоссе CLIP есть одна **неочевидная деталь**, без которой модель не учится.
Я **намеренно не рассказываю** про неё заранее и не добавил ее в семинар. Вы сначала напишете «наивную»
версию, при обучении наткнётесь на проблему, продиагностируете её по **норме
градиентов** и величине логитов, подсмотрите, как эту проблему решают в реальных
реализациях CLIP — и догадаетесь сами. Полный разбор «почему» ждёт вас в конце
ноутбука (разверните уже после того, как разберётесь).

Оценивается: **100 баллов** (2 задачи по 50).

## 1. Теория: контрастивный лосс CLIP

CLIP учит общее пространство для картинок и текстов так, чтобы эмбеддинги
**совпадающей** пары (картинка ↔ её подпись) были близки, а всех остальных
(негативных) пар в батче — далеки. Это симметричный InfoNCE:

$$\mathcal{L} = \tfrac{1}{2}\big(\mathcal{L}_{\text{img}\to\text{txt}} +
\mathcal{L}_{\text{txt}\to\text{img}}\big),$$

где каждое слагаемое — обычная кросс-энтропия по строкам матрицы похожестей
(правильный класс для строки $i$ — это столбец $i$, т.е. её пара).

Перед этим эмбеддинги **L2-нормируются**, поэтому их скалярное произведение — это
косинусная похожесть в диапазоне $[-1, 1]$. Самый наивный вариант — взять эти
косинусы прямо как логиты для кросс-энтропии. Так и поступим… для начала.

In [ ]:
# Установка зависимостей (Colab)
!pip install -q transformers datasets torch torchvision pillow

## 📝 Задача 1 (50 баллов): логиты CLIP + симметричный лосс

Реализуйте в клетке ниже две функции:

1. `compute_clip_logits(image_embeds, text_embeds, logit_scale)` — L2-нормируйте
   эмбеддинги и посчитайте попарные косинусные похожести. В сигнатуре есть
   `logit_scale` — обучаемый скаляр-«температура»; включите его в логиты. **Как
   именно** — деталь, в которой предстоит разобраться в Задаче 2; если сомневаетесь,
   начните с самого простого варианта и вернитесь сюда позже.
2. `clip_loss(logits_per_text)` — симметричная кросс-энтропия (используйте данную
   функцию `contrastive_loss`).

Функция `contrastive_loss` уже реализована — она задаёт, что правильная пара для
строки $i$ это столбец $i$.

In [ ]:
import torch
import torch.nn as nn


def compute_clip_logits(image_embeds: torch.Tensor,
                        text_embeds: torch.Tensor,
                        logit_scale: torch.Tensor):
    """Похожести «картинка ↔ текст» — логиты для контрастивного лосса CLIP.

    Шаги:
      1) L2-нормируйте image_embeds и text_embeds по последней оси.
      2) Посчитайте попарные косинусные похожести (матрица сходств).
      3) logit_scale — обучаемый скаляр-«температура». Включите его в логиты.
         КАК именно — та самая неочевидная деталь, в которой предстоит
         разобраться при обучении (Задача 2): начните с простого варианта,
         посмотрите на поведение градиентов и при необходимости вернитесь
         сюда. Если застрянете — подсмотрите, как формируют логиты реальные
         реализации CLIP (см. ссылки в блоке диагностики Задачи 2).

    Верните (logits_per_image, logits_per_text), где
    logits_per_image == logits_per_text.t().
    """
    raise NotImplementedError(
        "Реализуйте логиты CLIP: нормировка + косинусы + logit_scale")


def contrastive_loss(logits: torch.Tensor) -> torch.Tensor:
    # Cross-entropy where the correct class for row i is i (the matching pair).
    targets = torch.arange(len(logits), device=logits.device)
    return nn.functional.cross_entropy(logits, targets)


def clip_loss(logits_per_text: torch.Tensor) -> torch.Tensor:
    """Симметричный CLIP loss: среднее контрастивных лоссов
    «изображение → текст» и «текст → изображение».
    """
    raise NotImplementedError("Реализуйте симметричный CLIP loss")

### Базовая самопроверка Задачи 1
Проверяем форму логитов, симметрию и что ваш `clip_loss` совпадает с эталонной
реализацией из `transformers`. Этого достаточно, чтобы двигаться дальше — но
**достаточно ли этого, чтобы модель училась?** Узнаем в Задаче 2.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
img = torch.randn(8, 512); txt = torch.randn(8, 512)
logit_scale = nn.Parameter(torch.tensor(2.6592))
lpi, lpt = compute_clip_logits(img, txt, logit_scale)
assert lpt.shape == (8, 8), 'неверная форма логитов'
assert torch.allclose(lpi, lpt.t(), atol=1e-4), \
    'logits_per_image должен быть транспонированием logits_per_text'
# сверяем симметричный лосс с HuggingFace
try:  # transformers < 5.x
    from transformers.models.clip.modeling_clip import clip_loss as hf_clip_loss
except ImportError:  # transformers >= 5.x переименовал helper
    from transformers.models.clip.modeling_clip import image_text_contrastive_loss as hf_clip_loss
assert torch.allclose(clip_loss(lpt), hf_clip_loss(lpt), atol=1e-5), \
    'симметричный лосс не совпал с эталоном HF'
print('Базовая проверка пройдена. Достаточно ли этого для обучения? → Задача 2')

## 📝 Задача 2 (50 баллов): обучите мини-CLIP и сделайте zero-shot классификацию

Теперь соберём настоящий (но крошечный) CLIP: возьмём **замороженные** маленькие
предобученные энкодеры (картиночный `vit-tiny`, текстовый `bert-tiny`) и обучим
только две линейные проекции в общее пространство + параметр `logit_scale`,
используя ваши `compute_clip_logits` и `clip_loss`.

Датасет — небольшое подмножество **CIFAR-10**; подпись к картинке — `"a photo of a
{class}"`. После обучения проверим **zero-shot** классификацию: для тестовой
картинки выбираем класс, чья текстовая подпись ближе всего.

В цикле обучения мы логируем не только `loss`, но и **норму градиентов** обучаемых
параметров и текущий `logit_scale`. Следите за всеми тремя величинами.

In [ ]:
import torch, torch.nn as nn
from transformers import AutoModel, AutoTokenizer, AutoImageProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Самые маленькие предобученные энкодеры, какие удалось найти:
#   картинки: WinKawaks/vit-tiny-patch16-224 (~5.7M, hidden=192)
#   текст:    prajjwal1/bert-tiny           (~4.4M, hidden=128)
image_encoder = AutoModel.from_pretrained('WinKawaks/vit-tiny-patch16-224').to(DEVICE).eval()
image_processor = AutoImageProcessor.from_pretrained('WinKawaks/vit-tiny-patch16-224')

# У старого чекпойнта bert-tiny в config.json нет ключа `model_type`, а файлов
# токенизатора нет вовсе. transformers >= 5.x уже не угадывает это сам, поэтому
# грузим BERT явно (на старых версиях AutoModel/AutoTokenizer отработают как было).
try:
    text_encoder = AutoModel.from_pretrained('prajjwal1/bert-tiny').to(DEVICE).eval()
except (ValueError, KeyError):
    from transformers import BertModel, BertConfig
    bert_cfg = BertConfig.from_pretrained('prajjwal1/bert-tiny')
    text_encoder = BertModel.from_pretrained('prajjwal1/bert-tiny', config=bert_cfg).to(DEVICE).eval()
try:
    tokenizer = AutoTokenizer.from_pretrained('prajjwal1/bert-tiny')
except Exception:
    # bert-tiny использует словарь bert-base-uncased
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

IMAGE_HIDDEN = image_encoder.config.hidden_size  # 192
TEXT_HIDDEN = text_encoder.config.hidden_size    # 128

In [ ]:
class MiniCLIP(nn.Module):
    """Замороженные энкодеры + обучаемые проекции + обучаемый logit_scale."""
    def __init__(self, image_encoder, text_encoder, image_hidden, text_hidden, proj_dim=64):
        super().__init__()
        self.image_encoder = image_encoder
        self.text_encoder = text_encoder
        for p in self.image_encoder.parameters():
            p.requires_grad = False
        for p in self.text_encoder.parameters():
            p.requires_grad = False
        self.image_projection = nn.Linear(image_hidden, proj_dim, bias=False)
        self.text_projection = nn.Linear(text_hidden, proj_dim, bias=False)
        # logit_scale хранится в лог-пространстве, init = ln(1/0.07)
        self.logit_scale = nn.Parameter(torch.tensor(2.6592))

    def encode_image(self, pixel_values):
        with torch.no_grad():
            out = self.image_encoder(pixel_values=pixel_values)
            feats = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:, 0]
        return self.image_projection(feats)

    def encode_text(self, input_ids, attention_mask):
        with torch.no_grad():
            out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
            feats = out.last_hidden_state[:, 0]  # [CLS]
        return self.text_projection(feats)

    def forward(self, pixel_values, input_ids, attention_mask):
        image_embeds = self.encode_image(pixel_values)
        text_embeds = self.encode_text(input_ids, attention_mask)
        # клипуем logit_scale сверху, как в HF (ln(100))
        with torch.no_grad():
            self.logit_scale.clamp_(max=4.6052)
        return compute_clip_logits(image_embeds, text_embeds, self.logit_scale)

model = MiniCLIP(image_encoder, text_encoder, IMAGE_HIDDEN, TEXT_HIDDEN).to(DEVICE)

In [ ]:
from datasets import load_dataset

CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

# Берём небольшие подмножества, чтобы быстро обучаться в Colab.
# Свежие версии datasets уже не понимают короткий алиас 'cifar10' — берём
# канонический репозиторий 'uoft-cs/cifar10' (работает и на старых, и на новых).
try:
    train_ds = load_dataset('cifar10', split='train[:2000]')
    test_ds = load_dataset('cifar10', split='test[:1000]')
except Exception:
    train_ds = load_dataset('uoft-cs/cifar10', split='train[:2000]')
    test_ds = load_dataset('uoft-cs/cifar10', split='test[:1000]')

def collate(batch):
    images = [ex['img'] for ex in batch]
    labels = torch.tensor([ex['label'] for ex in batch])
    captions = [f'a photo of a {CLASSES[l]}' for l in labels]
    pixel_values = image_processor(images, return_tensors='pt')['pixel_values']
    tok = tokenizer(captions, padding=True, return_tensors='pt')
    return pixel_values, tok['input_ids'], tok['attention_mask'], labels

from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate)

In [ ]:
# Обучаем ТОЛЬКО проекции и logit_scale (энкодеры заморожены).
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.Adam(params, lr=1e-3)

model.train()
for epoch in range(3):
    total = 0.0; grad_total = 0.0; nb = 0
    for pixel_values, input_ids, attention_mask, _ in train_loader:
        pixel_values = pixel_values.to(DEVICE)
        input_ids = input_ids.to(DEVICE); attention_mask = attention_mask.to(DEVICE)
        opt.zero_grad()
        _, logits_per_text = model(pixel_values, input_ids, attention_mask)
        loss = clip_loss(logits_per_text)
        loss.backward()
        # ДИАГНОСТИКА: норма градиентов обучаемых параметров
        grad_norm = torch.sqrt(sum((p.grad.detach() ** 2).sum()
                                   for p in params if p.grad is not None))
        opt.step()
        total += loss.item(); grad_total += grad_norm.item(); nb += 1
    print(f'epoch {epoch}: loss={total/nb:.4f}  grad_norm={grad_total/nb:.2e}  '
          f'logit_scale={model.logit_scale.item():.3f}')

## 🔬 Модель не учится? Диагностика (разберитесь сами)

Если `loss` почти не падает, `grad_norm` крошечный, а zero-shot accuracy ниже
(на уровне случайной ~0.1) — это **не баг кода**, а та самая неочевидная деталь
лосса. Прежде чем читать разбор в конце, разберитесь сами:

1. **Посмотрите на величину логитов.** Распечатайте `logits_per_text` после
   `compute_clip_logits`. В каком они диапазоне? (Подсказка: эмбеддинги
   L2-нормированы, значит косинусы лежат в $[-1, 1]$.)
2. **Подумайте про softmax.** Насколько «острым» может быть `softmax` по $N$
   классам, если логиты зажаты в $[-1, 1]$? Что это означает для градиентов
   кросс-энтропии?
3. **Сравните норму градиентов** (`grad_norm` уже выводится) до и после вашей
   правки — это объективный сигнал, что что-то поменялось.
4. **Посмотрите внимательно, как это решают настоящие CLIP'и.**
   - [Статья CLIP](https://arxiv.org/pdf/2103.00020) - Псевдокод `Figure 3`
   - HF `modeling_clip.py`:
     https://github.com/huggingface/transformers/blob/v4.39.3/src/transformers/models/clip/modeling_clip.py#L1132
   - `config.json` → `logit_scale_init_value`:
     https://huggingface.co/openai/clip-vit-base-patch32/blob/main/config.json#L7

Догадались? Поправьте `compute_clip_logits`, перезапустите обучение и убедитесь,
что `loss` начинает падать, `grad_norm` подрос, а zero-shot заметно лучше
случайного.

### Финальная самопроверка Задачи 1 (повторяет автогрейдер)
После того как вы починили `compute_clip_logits`, эта клетка проверяет ровно то,
что проверяет автогрейдер: нормировку, **что `logit_scale` действительно
масштабирует логиты**, совпадение лосса с HF и поток градиента в `logit_scale`.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
img = torch.randn(8, 512); txt = torch.randn(8, 512)
ls = nn.Parameter(torch.tensor(2.6592))
lpi, lpt = compute_clip_logits(img, txt, ls)
assert lpt.shape == (8, 8) and torch.allclose(lpi, lpt.t(), atol=1e-4)
# нормировка: при logit_scale=0 косинус эмбеддинга с собой = 1
_, diag_lpt = compute_clip_logits(img, img, nn.Parameter(torch.tensor(0.0)))
assert torch.allclose(torch.diagonal(diag_lpt), torch.ones(8), atol=1e-4), \
    'эмбеддинги не нормированы?'
# logit_scale действительно масштабирует логиты
_, lpt0 = compute_clip_logits(img, txt, nn.Parameter(torch.tensor(0.0)))
ratio = lpt.abs().mean() / lpt0.abs().mean()
assert ratio > 5.0, f'logit_scale почти не влияет на логиты (ratio={ratio:.2f})'
# симметричный лосс совпадает с HF
try:  # transformers < 5.x
    from transformers.models.clip.modeling_clip import clip_loss as hf_clip_loss
except ImportError:  # transformers >= 5.x переименовал helper
    from transformers.models.clip.modeling_clip import image_text_contrastive_loss as hf_clip_loss
assert torch.allclose(clip_loss(lpt), hf_clip_loss(lpt), atol=1e-5)
# градиент течёт в logit_scale
loss = clip_loss(lpt); loss.backward()
assert ls.grad is not None and ls.grad.abs() > 0
print('✅ Финальная проверка пройдена — совпадает с автогрейдером Задачи 1')

In [ ]:
# Zero-shot классификация: класс = ближайшая текстовая подпись.
import torch
model.eval()
prompts = [f'a photo of a {c}' for c in CLASSES]
tok = tokenizer(prompts, padding=True, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    text_embeds = model.encode_text(tok['input_ids'], tok['attention_mask'])
    text_embeds = text_embeds / text_embeds.norm(p=2, dim=-1, keepdim=True)

correct = 0; seen = 0
test_loader = DataLoader(test_ds, batch_size=128, collate_fn=collate)
with torch.no_grad():
    for pixel_values, _, _, labels in test_loader:
        img_embeds = model.encode_image(pixel_values.to(DEVICE))
        img_embeds = img_embeds / img_embeds.norm(p=2, dim=-1, keepdim=True)
        logits = model.logit_scale.exp() * img_embeds @ text_embeds.t()
        pred = logits.argmax(dim=1).cpu()
        correct += (pred == labels).sum().item(); seen += len(labels)
print(f'zero-shot accuracy: {correct/seen:.3f}  (случайная ~0.1)')
assert correct/seen > 0.2, 'accuracy на уровне случайной — проверьте compute_clip_logits!'

### Финальная самопроверка Задачи 2 (повторяет автогрейдер)
Эта клетка запускает **ровно тот же тест**, что и автогрейдер Задачи 2
(`tests/clip/test_clip_training.py`): на крошечном синтетическом датасете обучает
две линейные проекции + `logit_scale` поверх ваших `compute_clip_logits` и
`clip_loss` и проверяет, что обучение **сходится** (лосс падает ниже 0.5, in-batch
retrieval ≥ 0.9) и **детерминированно**.

Тест самодостаточный и быстрый (CPU, пара секунд) — он не использует ни CIFAR, ни
ваш `MiniCLIP` выше, поэтому проходит, **только если в `compute_clip_logits` есть
та самая деталь с `logit_scale.exp()`**. Если она забыта — лосс застрянет у ~2.5.

In [ ]:
# Тот же самый тест, что гоняет автогрейдер Задачи 2 (tests/clip/test_clip_training.py),
# но прямо здесь, на CPU, за пару секунд. Он НЕ зависит от CIFAR/MiniCLIP выше:
# строит крошечный синтетический мультимодальный датасет и обучает две линейные
# проекции + logit_scale поверх ВАШИХ compute_clip_logits и clip_loss.
# Решающая проверка — итоговый loss < 0.05. Она отделяет правильный logit_scale.exp()
# от двух «почти верных» вариантов:
#   * без масштабирования (scale=1): косинусы в [-1, 1], softmax не заостряется,
#     loss застревает у ~2.5;
#   * умножение на СЫРОЙ logit_scale (без .exp()): параметр обучаемый и клампится
#     до 4.6052, поэтому масштаб упирается в ~4.6 и loss застревает у ~0.3 —
#     лосс вроде «падает» и retrieval идеальный, но детали exp() нет.
# Только exp() разгоняет масштаб до exp(4.6052)=100 и роняет loss почти в 0.
import torch
import torch.nn as nn

LOGIT_SCALE_INIT = 2.6592


def _make_synthetic_dataset(n=32, latent_dim=16, img_feat=48, txt_feat=40, noise=0.05, seed=0):
    g = torch.Generator().manual_seed(seed)
    z = torch.randn(n, latent_dim, generator=g)            # общий «концепт» на пару
    A = torch.randn(latent_dim, img_feat, generator=g)     # фиксированный «картиночный энкодер»
    B = torch.randn(latent_dim, txt_feat, generator=g)     # фиксированный «текстовый энкодер»
    image_feats = z @ A + noise * torch.randn(n, img_feat, generator=g)
    text_feats = z @ B + noise * torch.randn(n, txt_feat, generator=g)
    return image_feats, text_feats


class _TinyCLIP(nn.Module):
    def __init__(self, img_feat, txt_feat, proj_dim=16):
        super().__init__()
        self.img_proj = nn.Linear(img_feat, proj_dim, bias=False)
        self.txt_proj = nn.Linear(txt_feat, proj_dim, bias=False)
        self.logit_scale = nn.Parameter(torch.tensor(LOGIT_SCALE_INIT))

    def forward(self, image_feats, text_feats):
        image_embeds = self.img_proj(image_feats)
        text_embeds = self.txt_proj(text_feats)
        return compute_clip_logits(image_embeds, text_embeds, self.logit_scale)


def _train_and_eval(steps=400, lr=0.05, seed=0):
    torch.manual_seed(seed)
    image_feats, text_feats = _make_synthetic_dataset(seed=seed)
    model = _TinyCLIP(image_feats.shape[1], text_feats.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    first_loss = None
    for _ in range(steps):
        opt.zero_grad()
        _, logits_per_text = model(image_feats, text_feats)
        loss = clip_loss(logits_per_text)
        loss.backward()
        opt.step()
        # держим logit_scale в диапазоне CLIP (<= ln(100)), как делает HF
        with torch.no_grad():
            model.logit_scale.clamp_(max=4.6052)
        if first_loss is None:
            first_loss = loss.item()

    with torch.no_grad():
        logits_per_image, _ = model(image_feats, text_feats)
        n = logits_per_image.shape[0]
        targets = torch.arange(n)
        img_to_txt = (logits_per_image.argmax(dim=1) == targets).float().mean().item()
        txt_to_img = (logits_per_image.argmax(dim=0) == targets).float().mean().item()
    return first_loss, loss.item(), img_to_txt, txt_to_img


# --- сходимость (повторяет test_contrastive_training_converges) ---
first_loss, last_loss, img_to_txt, txt_to_img = _train_and_eval()
assert last_loss < first_loss, f'лосс не уменьшился ({first_loss:.3f} -> {last_loss:.3f})'
assert img_to_txt >= 0.9, f'retrieval картинка->текст слишком низкий: {img_to_txt:.2f}'
assert txt_to_img >= 0.9, f'retrieval текст->картинка слишком низкий: {txt_to_img:.2f}'
assert last_loss < 0.05, (
    f'итоговый лосс {last_loss:.3f} не упал почти в ноль. Косинусы нужно масштабировать '
    'на logit_scale.exp(), а не на сырой logit_scale (тогда лосс застревает у ~0.3) '
    'и не оставлять без масштаба.'
)

# --- детерминированность (повторяет test_training_is_deterministic) ---
a = _train_and_eval(seed=0)
b = _train_and_eval(seed=0)
assert abs(a[1] - b[1]) < 1e-6, 'обучение должно быть детерминированным при одном seed'

print(f'✅ Финальная проверка Задачи 2 пройдена — совпадает с автогрейдером\n'
      f'   loss: {first_loss:.3f} -> {last_loss:.3f} | '
      f'retrieval img->txt={img_to_txt:.2f}, txt->img={txt_to_img:.2f}')